In [ ]:
!pip install transformers datasets accelerate scikit-learn -q

In [ ]:
import json
with open("severity_colab_export.json") as f:
    data = json.load(f)
LABELS = data["labels"]
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}
print({k: len(data[k]) for k in ("train", "dev", "test")})

{'train': 6721, 'dev': 961, 'test': 1921}


In [ ]:
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 512
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(rows):
    return Dataset.from_dict({"text": [r["text"] for r in rows],
                              "labels": [r["label"] for r in rows]})

raw = DatasetDict({k: to_ds(data[k]) for k in ("train", "dev", "test")})

def encode(batch):
    enc = tokenizer(batch["text"], add_special_tokens=False, truncation=False)
    input_ids = []
    for ids in enc["input_ids"]:
        room = MAX_LEN - 2
        if len(ids) > room:                      # too long: keep the start and the end
            head = room // 2
            ids = ids[:head] + ids[-(room - head):]
        input_ids.append([tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id])
    return {"input_ids": input_ids, "attention_mask": [[1] * len(x) for x in input_ids]}

tokenized = raw.map(encode, batched=True, remove_columns=["text"])
lens = np.array([len(x) for x in tokenized["train"]["input_ids"]])
print(f"median length {int(np.median(lens))} tokens; "
      f"{np.mean(lens >= MAX_LEN):.1%} of training narratives were cut to {MAX_LEN} (start + end kept)")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/6721 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (659 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/961 [00:00<?, ? examples/s]

Map:   0%|          | 0/1921 [00:00<?, ? examples/s]

median length 178 tokens; 5.9% of training narratives were cut to 512 (start + end kept)


In [ ]:
import torch
from transformers import (AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, EarlyStoppingCallback)
from sklearn.metrics import f1_score, accuracy_score

counts = np.bincount([r["label"] for r in data["train"]], minlength=len(LABELS))
weights = (counts.sum() / (len(LABELS) * counts)) ** 0.5     # square-root inverse frequency
class_weights = torch.tensor(weights, dtype=torch.float)
print("train counts:", dict(zip(LABELS, counts.tolist())))
print("loss weights:", dict(zip(LABELS, weights.round(2).tolist())))

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def macro_f1(y_true, y_pred):
    present = sorted(set(y_true))          # macro over classes that occur in y_true
    return f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"macro_f1": macro_f1(labels, preds), "accuracy": accuracy_score(labels, preds)}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir="./severity_checkpoint",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=2,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["dev"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()

train counts: {'minor': 5707, 'moderate': 692, 'severe': 229, 'critical': 93}
loss weights: {'minor': 0.54, 'moderate': 1.56, 'severe': 2.71, 'critical': 4.25}


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2526, training_loss=0.26263616720740984, metrics={'train_runtime': 671.3723, 'train_samples_per_second': 80.087, 'train_steps_per_second': 5.017, 'total_flos': 4952847568080048.0, 'train_loss': 0.26263616720740984, 'epoch': 6.0})

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

pred_out = trainer.predict(tokenized["test"])
y_true = pred_out.label_ids
y_pred = pred_out.predictions.argmax(-1)
test_commodity = np.array([r["commodity_type"] for r in data["test"]])
ALL = list(range(len(LABELS)))

print(classification_report(y_true, y_pred, labels=ALL, target_names=LABELS, digits=3, zero_division=0))
print(f"Overall macro-F1 {macro_f1(y_true, y_pred):.3f}   accuracy {accuracy_score(y_true, y_pred):.3f}")
print("\nConfusion matrix (rows = true, columns = predicted):")
print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=ALL), index=LABELS, columns=LABELS).to_string())

print("\nPer commodity (macro-F1 over classes present; small classes are very noisy):")
for c in sorted(set(test_commodity)):
    m = test_commodity == c
    support = {LABELS[k]: int((y_true[m] == k).sum()) for k in ALL}
    print(f"  {c:<28} n={int(m.sum()):<5} macro-F1 {macro_f1(y_true[m], y_pred[m]):.3f}   support {support}")

              precision    recall  f1-score   support

       minor      0.984     0.991     0.987      1630
    moderate      0.787     0.879     0.831       198
      severe      0.816     0.470     0.596        66
    critical      0.842     0.593     0.696        27

    accuracy                          0.956      1921
   macro avg      0.857     0.733     0.777      1921
weighted avg      0.956     0.956     0.954      1921

Overall macro-F1 0.777   accuracy 0.956

Confusion matrix (rows = true, columns = predicted):
          minor  moderate  severe  critical
minor      1616         9       5         0
moderate     19       174       2         3
severe        5        30      31         0
critical      3         8       0        16

Per commodity (macro-F1 over classes present; small classes are very noisy):
  gas_distribution             n=316   macro-F1 0.749   support {'minor': 118, 'moderate': 121, 'severe': 56, 'critical': 21}
  gas_transmission_gathering   n=410   macro-F1

In [ ]:
# ---- Cell 5b (optional): how well does each model do on cases where the text actually shows the cause? ----
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

DEATH = re.compile(r"\b(fatal\w*|death\w*|died|dies|dead|killed|deceased|passed away|"
                   r"lost (?:his|her|their) li(?:fe|ves)|succumbed)\b", re.I)
INJURY = re.compile(r"\b(injur\w*|hospital\w*|ambulance|paramedic\w*|treated for)\b", re.I)
NEGATION = re.compile(r"\b(no|not|without|zero|neither|nor|never)\b[^.;,]{0,40}", re.I)

def has(pattern, text):
    return bool(pattern.search(NEGATION.sub(" ", text)))

test_texts = [r["text"] for r in data["test"]]
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform([r["text"] for r in data["train"]])
base = LogisticRegression(C=30, class_weight="balanced", max_iter=3000).fit(
    Xtr, [r["label"] for r in data["train"]])
base_pred = base.predict(vec.transform(test_texts))

def recall_on(idx, pred, cls):
    return float(np.mean([pred[i] == cls for i in idx])) if idx else float("nan")

for cls, pat, name in ((3, DEATH, "death word"), (2, INJURY, "injury word")):
    idx = [i for i in range(len(y_true)) if y_true[i] == cls]
    vis = [i for i in idx if has(pat, test_texts[i])]
    vis_set = set(vis)
    hid = [i for i in idx if i not in vis_set]
    print(f"\n{LABELS[cls]}: {len(vis)} test narratives with a {name}, {len(hid)} without")
    for model_name, pred in (("TF-IDF", base_pred), ("DistilBERT", y_pred)):
        print(f"   {model_name:<11} recall with the word {recall_on(vis, pred, cls):.0%}"
              f"   without {recall_on(hid, pred, cls):.0%}")


critical: 16 test narratives with a death word, 11 without
   TF-IDF      recall with the word 31%   without 9%
   DistilBERT  recall with the word 100%   without 0%

severe: 47 test narratives with a injury word, 19 without
   TF-IDF      recall with the word 62%   without 16%
   DistilBERT  recall with the word 64%   without 5%


In [ ]:
trainer.save_model("./phmsa_severity_model")
tokenizer.save_pretrained("./phmsa_severity_model")

!zip -r phmsa_severity_model.zip phmsa_severity_model
from google.colab import files
files.download("phmsa_severity_model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: phmsa_severity_model/ (stored 0%)
  adding: phmsa_severity_model/tokenizer.json (deflated 71%)
  adding: phmsa_severity_model/training_args.bin (deflated 53%)
  adding: phmsa_severity_model/config.json (deflated 51%)
  adding: phmsa_severity_model/model.safetensors (deflated 8%)
  adding: phmsa_severity_model/tokenizer_config.json (deflated 43%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>